In [1]:
# Importo las librerías para conectarme y trabajar con dataframes
from sqlalchemy import create_engine
import pandas as pd
import pyodbc
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv

# Cargo las variables de conexión desde el archivo .env (override por si el kernel ya tenía valores cacheados)
load_dotenv(override=True)

True

In [2]:
# Configuro la conexión a PostgreSQL de origen usando las variables del .env
usuario = os.getenv("PG_USUARIO")
password = os.getenv("PG_PASSWORD")
servidor = os.getenv("PG_SERVIDOR")
puerto = os.getenv("PG_PUERTO")
base_datos = os.getenv("PG_BASE_DATOS")

cadena = (
    f"postgresql+psycopg2://{usuario}:{password}@"
    f"{servidor}:{puerto}/{base_datos}"
)

engine_origen = create_engine(cadena)

try:
    with engine_origen.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [3]:
# Extraigo los datos de ventas desde la base de datos origen
consulta_venta = "SELECT * FROM tienda.venta"
df_venta = pd.read_sql_query(consulta_venta, engine_origen)

In [4]:
# Configuro la conexión al stage en SQL Server usando las variables del .env
servidor_sqlserver = os.getenv("SQLSERVER_SERVIDOR")
usuario_sqlserver = os.getenv("SQLSERVER_USUARIO")
password_sqlserver = os.getenv("SQLSERVER_PASSWORD")
base_datos_stage = os.getenv("SQLSERVER_BASE_DATOS")
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")
driver_sqlserver = os.getenv("SQLSERVER_DRIVER")

cadena_sqlserver = (
    f"mssql+pyodbc://{usuario_sqlserver}:{quote_plus(password_sqlserver)}@"
    f"{servidor_sqlserver}/{base_datos_stage}?driver={quote_plus(driver_sqlserver)}"
)

engine_stage = create_engine(cadena_sqlserver)
try:
    with engine_stage.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [5]:
# Agrego la fecha y hora de carga al dataframe de ventas
df_venta["fecha_carga"] = pd.Timestamp.now()

In [6]:
# Reviso una muestra de ventas antes de cargarla
df_venta.head()

,venta_id,cliente_id,empleado_id,sucursal_id,metodo_pago_id,canal_venta_id,fecha_venta,subtotal,descuento,impuesto,total,estado,fecha_creacion,fecha_modificacion,fecha_carga
0,1,663.0,NaN,1,1,2,2023-10-08 17:44:13,442.9,22.15,54.70,475.45,COMPLETADA,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:14:43.611394
1,2,NaN,NaN,1,4,2,2023-07-16 13:46:24,685.7,56.39,81.81,711.12,COMPLETADA,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:14:43.611394
2,3,197.0,20.0,4,4,1,2023-07-20 22:42:22,836.4,8.64,107.61,935.37,PENDIENTE,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:14:43.611394
3,4,740.0,2.0,1,2,1,2023-11-27 15:00:31,380.0,0.00,49.40,429.40,COMPLETADA,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:14:43.611394
4,5,380.0,21.0,4,4,1,2023-11-07 16:32:00,2156.4,64.64,271.93,2363.69,COMPLETADA,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-30 14:14:43.611394


In [7]:
# Cargo las ventas en la tabla del stage
df_venta.to_sql(
    name="venta",
    con=engine_stage,
    schema=esquema_stage,
    if_exists="append",
    index=False
)

131

In [8]:
# Verifico el contenido de ventas en el stage
consulta_stage = f"SELECT * FROM {esquema_stage}.venta"
df_venta_stage = pd.read_sql_query(consulta_stage, engine_stage)
df_venta_stage.head()

,venta_id,cliente_id,empleado_id,sucursal_id,metodo_pago_id,canal_venta_id,fecha_venta,subtotal,descuento,impuesto,total,estado,fecha_creacion,fecha_modificacion,fecha_carga
0,1,663.0,NaN,1,1,2,2023-10-08 17:44:13,442.9,22.15,54.70,475.45,COMPLETADA,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:14:43.610
1,2,NaN,NaN,1,4,2,2023-07-16 13:46:24,685.7,56.39,81.81,711.12,COMPLETADA,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:14:43.610
2,3,197.0,20.0,4,4,1,2023-07-20 22:42:22,836.4,8.64,107.61,935.37,PENDIENTE,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:14:43.610
3,4,740.0,2.0,1,2,1,2023-11-27 15:00:31,380.0,0.00,49.40,429.40,COMPLETADA,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:14:43.610
4,5,380.0,21.0,4,4,1,2023-11-07 16:32:00,2156.4,64.64,271.93,2363.69,COMPLETADA,2026-09-16 14:22:49.160,2026-09-16 14:22:49.160,2026-09-30 14:14:43.610
